# 1021 34일차

## 1. 그래프 그림 출력

`draw_mermaid_png()`로 받은 PNG 바이트를 파일로 저장하지 않고 바로 띄우는 방법 (`lg07_ToolNode_graph02`)

1. `BytesIO(바이트)` : 바이트를 파일처럼 읽게 감쌈
2. `Image.open(...)` : PIL 이미지로 열기
3. `plt.imshow(img)` → `plt.show()` : matplotlib으로 출력

- `plt.axis('off')` : 축 눈금을 지움
- Jupyter에서는 `plt.imshow`까지만 해도 그림이 나옴

## 2. 메모리

이전 실행의 State를 저장해 두고 다음 실행에 이어 쓰는 기능 (`lg08_Memory01`, `lg08_Memory02`)

메모리가 없으면 `invoke`마다 State가 새로 시작돼서, 두 번째 질문 때 `messages`에는 그 질문 하나만 들어감. 앞 대화를 참고해야 하는 질문에 답하지 못함

1. checkpointer : 노드가 실행될 때마다 State를 저장하는 객체
2. `InMemorySaver()` : State를 RAM에 저장하는 checkpointer. 프로그램이 끝나면 사라짐
3. `compile(checkpointer=memory)` : 그래프에 checkpointer를 연결
4. `thread_id` : 대화를 구분하는 이름. `config={"configurable": {"thread_id": "user01"}}`를 `invoke`에 같이 넘김

- 같은 `thread_id`면 저장된 State에 이어서 실행하고, 다른 `thread_id`면 빈 State에서 시작
- 저장된 `messages` 뒤에 새 질문이 `add_messages`로 붙어서 모델이 앞 대화까지 같이 받음

## 3. fan-out / fan-in

한 노드에서 여러 노드로 갈라져 같이 실행하고(fan-out), 그 결과를 한 노드에서 합치는(fan-in) 구조 (`lg09_fanout_fanin1`, `lg09_fanout_fanin2_concat`)

서로 기다릴 필요가 없는 작업(웹 검색, 모델 분석 등)을 한 단계에서 같이 실행하고 결과를 한 번에 모으려고 씀

1. fan-out : 한 노드에서 `add_edge`를 여러 개 내면 도착 노드들이 같은 단계에서 같이 실행됨
2. fan-in : 여러 노드에서 한 노드로 `add_edge`를 내면 앞 노드들이 끝난 다음 단계에서 그 노드가 실행됨

- 갈라진 노드는 각자 다른 State 키에 결과를 쓰고, 합치는 노드가 그 키들을 읽어 최종 결과를 만듦
- 합친 결과를 `messages`에 `("assistant", 문자열)`로 돌려주면 `add_messages`가 `AIMessage`로 바꿔서 붙임

#### cf) 노드가 돌려주는 키

- 같은 단계에서 실행되는 노드들이 리듀서 없는 같은 키를 바꾸면 `InvalidUpdateError`. 리듀서 없는 키는 한 단계에 값 하나만 받음
- 노드마다 다른 키를 쓰거나, 그 키에 리듀서를 붙임
- 노드가 돌려준 키 이름이 State에 없으면 에러 없이 무시됨 (`message` ≠ `messages`)